# Cross-Model Comparison: Accessibility Knowledge

Loads saved results from GPT-2 Medium, Large, and XL accessibility notebooks.
The big question: at what scale does the model *actually know* accessibility facts,
and does the retrieval mechanism change with scale?

**Run the individual model accessibility notebooks first.**

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
from src.viz import plot_phase_comparison

## WCAG: When does each model find "Web"?

GPT-2 Medium couldn't do this — final prediction was a quotation mark.
Does Large or XL break through?

In [ ]:
models_wcag = {}
for model_name in ["gpt2-medium", "gpt2-large", "gpt2-xl"]:
    try:
        df = pd.read_csv(f"../results/{model_name}/wcag-logit-lens.csv")
        models_wcag[model_name] = df
        print(f"Loaded {model_name}")
    except FileNotFoundError:
        print(f"Not yet: {model_name}")

In [ ]:
if models_wcag:
    plot_phase_comparison(models_wcag, metric="target_rank")
    plot_phase_comparison(models_wcag, metric="target_prob")

## Head Comparison: Accessibility vs Factual Recall

Compare which heads fire for WCAG vs which heads fire for geography.
Are they the same circuits or different?

In [ ]:
for model_name in ["gpt2-medium", "gpt2-large", "gpt2-xl"]:
    print(f"\n{'='*50}")
    print(f"{model_name}")
    print(f"{'='*50}")
    
    for domain, label in [("paris", "Paris (factual)"), ("wcag", "Web (accessibility)")]:
        try:
            df = pd.read_csv(f"../results/{model_name}/{domain}-heads.csv")
            top = df.nlargest(5, "abs_logit")[["layer", "head", "logit"]]
            print(f"\n  Top 5 heads → {label}:")
            print(f"  {top.to_string(index=False)}")
        except FileNotFoundError:
            print(f"\n  Not yet: {domain}")

## Decomposition Comparison: Attention vs MLP Balance

Does accessibility knowledge lean more on MLPs than factual recall?

In [ ]:
for model_name in ["gpt2-medium", "gpt2-large", "gpt2-xl"]:
    print(f"\n{model_name}:")
    for domain, label in [("paris", "Paris"), ("wcag", "Web")]:
        try:
            df = pd.read_csv(f"../results/{model_name}/{domain}-decomposition.csv")
            total_attn = df["attn_logit"].sum()
            total_mlp = df["mlp_logit"].sum()
            ratio = total_attn / total_mlp if total_mlp != 0 else float('inf')
            print(f"  {label}: attn={total_attn:.1f}, mlp={total_mlp:.1f}, ratio={ratio:.2f}")
        except FileNotFoundError:
            print(f"  {label}: not yet")